# April 05
我把utils.py里面的Reward Signals给重新整理了下然后现在有6种不同的signals.
直接用train_ray_selfplay.py 去从头训练这个新的Reward shape

Submit 了2个Train 一个CPU版本一个GPU版本
等看看训练结果，如果GPU没问题以后就只用GPU训练就好了。

NEXT:
出结果后和ceia_baseline_agent 打个比赛看看效果。

In [ ]:
#SBATCH --job-name=GPU_S2_selfplay
#SBATCH -N1 --ntasks-per-node=12 --gres=gpu:V100:1
#SBATCH --mem-per-cpu=6G
#SBATCH -t12:00:00
#SBATCH --output=soccerstwos-%j.out
#SBATCH --mail-type=END,FAIL
#SBATCH --mail-user=frank.yang@gatech.edu


source ~/miniconda3/etc/profile.d/conda.sh
conda activate soccertwos

# Fix protobuf version conflict with ray==1.4.0
export PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION=python
pip install "numpy==1.23.5" -q  # fix numpy version conflict with ray==1.4.0

# Run the training script
/storage/ice1/7/4/fyang365/.conda/conda_envs/soccertwos/bin/python train_ray_selfplay.py


# April 06
GPU在这个项目里只是把 PPO 的模型更新（forward/backward pass）放到 GPU 上，但 soccer-twos 的环境是 Unity 进程，仿真本身无法用 GPU 加速，而且 MLP 太小，GPU 收益可以忽略不计。
之后训练直接用CPU的就行。


### 当前脚本不支持多节点
train_ray_selfplay.py:40
ray.init()  # 只初始化本地 Ray，不知道其他节点
多节点需要改成 ray.init(address='auto') 并在 Slurm 脚本里手动启动 Ray 集群（head + workers），比较复杂。

### 修复不能继续训练的问题
问题：checkpoint 序列化时的类型信息丢失。
具体链路是这样的：
Ray 保存 checkpoint 时，把 optimizer state dict（里面是 torch tensor）先转成 numpy array，再用 pickle 序列化存盘
某些 tensor（比如 Adam 的 step 计数器，是一个标量 int64）在转换过程中被存成了 numpy.object_ 类型，而不是具体的数值类型
恢复时 Ray 想把这些 numpy array 转回 torch tensor，调用 torch.from_numpy()，但 torch 拒绝转换 object_ 类型

修复方式：
去到/storage/ice1/7/4/fyang365/.conda/conda_envs/soccertwos/lib/python3.8/site-packages/ray/rllib/utils/torch_utils.py
把def mapping(item)里面的
```
# Everything else: Convert to numpy, then wrap as torch tensor.
else:
    tensor = torch.from_numpy(np.asarray(item))
```
替换为：
```
# Everything else: Convert to numpy, then wrap as torch tensor.
else:
    arr = np.asarray(item)
    if arr.dtype == object:
        arr = np.array(arr.tolist(), dtype=np.float32)
    # Scalar values (like lr) should stay as Python scalars, not tensors
    if arr.ndim == 0:
        return arr.item()
    tensor = torch.from_numpy(arr)
```
